# Pipelines and ColumnTransformer

<div style="background-color:#1e293b;padding:15px;border-left:6px solid #38bdf8;color:#e2e8f0">

Until now, every preprocessing step has been your personal responsibility: fit the scaler on the training data, remember to reuse it on the test data, encode the categoricals the same way in both places, glue the arrays together in the right order. It works, and you have done it, but every step is a chance for a silent mistake.

In this notebook you meet scikit-learn's <b>ColumnTransformer</b> and <b>Pipeline</b>: one object that holds your entire preprocessing and your model, fits on a raw DataFrame, predicts on a raw DataFrame, and makes fit-on-train-only automatic instead of a discipline you must remember. This object is the standard shape of every model you will build from here on.

</div>

---
## Step 1: The Manual Way

The dataset is a familiar one: used-car listings with a mix of numeric features (`Year`, `KM_Driven`), nominal categories (`Brand`, `Fuel`, `Seller_Type`, `Transmission`), and one genuinely ordinal category (`Owner`). The target is the selling price, which is heavily right-skewed, so we model its log.

> **Data source:** Vehicle dataset from CarDekho (India), distributed with this course as `car_price.csv`. Download it from the course folder and place it in the same directory as this notebook. Prices are in Indian rupees.

Before the new tools, the full manual workflow one last time: three transformers, nine fit or transform calls, and a horizontal stack. Notice how much of it is bookkeeping.

In [49]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

RANDOM_SEED = 23

df = pd.read_csv('car_price.csv')
X = df.drop(columns=['Selling_Price'])
y = np.log1p(df['Selling_Price'])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
print(f"Train: {len(X_train)} rows, validation: {len(X_val)} rows")

num_cols = ['Year', 'KM_Driven']
onehot_cols = ['Brand', 'Fuel', 'Seller_Type', 'Transmission']
owner_order = ['Test Drive Car', 'First Owner', 'Second Owner', 'Third Owner', 'Fourth & Above Owner']

Train: 3472 rows, validation: 868 rows


In [50]:
# The manual workflow: every fit on train only, every transform repeated for validation
scaler = StandardScaler()
scaler.fit(X_train[num_cols])
X_num_train = scaler.transform(X_train[num_cols])
X_num_val = scaler.transform(X_val[num_cols])

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train[onehot_cols])
X_cat_train = ohe.transform(X_train[onehot_cols])
X_cat_val = ohe.transform(X_val[onehot_cols])

owner_enc = OrdinalEncoder(categories=[owner_order])
owner_enc.fit(X_train[['Owner']])
X_own_train = owner_enc.transform(X_train[['Owner']])
X_own_val = owner_enc.transform(X_val[['Owner']])

X_train_manual = np.hstack([X_num_train, X_cat_train, X_own_train])
X_val_manual = np.hstack([X_num_val, X_cat_val, X_own_val])

manual_model = Ridge(alpha=1.0).fit(X_train_manual, y_train)
manual_train = root_mean_squared_error(y_train, manual_model.predict(X_train_manual))
manual_val = root_mean_squared_error(y_val, manual_model.predict(X_val_manual))
print(f"Manual workflow: {X_train_manual.shape[1]} features, "
      f"train RMSE {manual_train:.4f}, val RMSE {manual_val:.4f} (log units)")

Manual workflow: 42 features, train RMSE 0.3783, val RMSE 0.3896 (log units)


> **Note:** It runs, and the discipline is correct: every `fit` sees training data only. But count the ways this can silently go wrong. Fit a transformer on the full dataset before splitting, and you have leakage. Call `fit_transform` instead of `transform` on the validation set, and the encoder relearns its vocabulary from data it should never learn from. Stack the arrays in a different order for train and validation, and the model reads columns in the wrong order. None of these mistakes crash. They just quietly change what your scores mean. And every new experiment repeats all of this code.

---
## Step 2: ColumnTransformer: Declare the Preprocessing Once

A `ColumnTransformer` replaces the manual calls with a declaration. You state which transformer handles which group of columns, once, and it routes the columns, fits every branch, and concatenates the results in a fixed order for you. Each branch gets a name you choose.

In [51]:
preprocess = ColumnTransformer(transformers=[
    ('num', RobustScaler(), num_cols),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), onehot_cols),
    ('owner', OrdinalEncoder(categories=[owner_order]), ['Owner']),
])

X_train_ct = preprocess.fit_transform(X_train)
print(f"Output shape: {X_train_ct.shape}")
print("First feature names:", list(preprocess.get_feature_names_out()[:5]))

Output shape: (3472, 42)
First feature names: ['num__Year', 'num__KM_Driven', 'onehot__Brand_Ambassador', 'onehot__Brand_Audi', 'onehot__Brand_BMW']


> **Note:** The declaration reads like a specification of your preprocessing: numbers scaled, nominal categories one-hot encoded with unknown-safe handling, `Owner` ordinal with its order stated explicitly. Two details worth registering. First, any column you do not mention (here `Model`, which we exclude for its huge number of distinct values) is **dropped by default** (`remainder='drop'`). That is convenient when the exclusion is deliberate and a silent trap when it is not, so make a habit of checking the output width (42 columns here) against what you expect. Second, `get_feature_names_out()` prefixes every feature with its branch name (`num__`, `onehot__`), so you can always trace a column back to the transformer that produced it.

---
## Step 3: Pipeline: Preprocessing and Model, One Object

A `Pipeline` chains steps into a single estimator: here, the `ColumnTransformer` followed by a model. The whole thing fits with one call on the raw DataFrame and predicts with one call on a raw DataFrame.

In [52]:
pipe = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', Lasso(alpha=0.001, max_iter=50000)),
])

pipe.fit(X_train, y_train)

pipe_train = root_mean_squared_error(y_train, pipe.predict(X_train))
pipe_val = root_mean_squared_error(y_val, pipe.predict(X_val))
print(f"Pipeline: train RMSE {pipe_train:.4f}, val RMSE {pipe_val:.4f} (log units)")

lasso_pipe = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', Lasso(alpha=0.001, max_iter=50000)),
])
lasso_pipe.fit(X_train, y_train)

print(root_mean_squared_error(y_val, lasso_pipe.predict(X_val)))
print((lasso_pipe.named_steps['model'].coef_ == 0).sum())


Pipeline: train RMSE 0.3849, val RMSE 0.4005 (log units)
0.4004523968500745
15


> **Note:** Identical numbers to the manual workflow (train 0.3783, validation 0.3896): same math, a fraction of the moving parts. Look at what `fit` did. It called `fit_transform` on the ColumnTransformer using only `X_train`, then fit the Ridge on the result. When you call `predict(X_val)`, the pipeline runs `transform` (never `fit`) with the train-fitted transformers. The fit-on-train-only rule is no longer something you remember; it is something the object enforces. There is no code path through a fitted pipeline that lets validation data leak into preprocessing statistics.
>
> For scale: an RMSE of 0.39 in log units means predictions are typically off by a factor of about e^0.39, roughly 1.5x in either direction. Used-car prices are noisy.

> **Note:** From this point in the course onward, project submissions are expected to be built as Pipelines. This is not extra formality: it is the difference between scores that are trustworthy by construction and scores you have to re-check by hand.

---
## Step 4: What a Pipeline Gives You

Three quick demonstrations of why this structure is useful in practice.

### Predict for a raw, human-readable example

No manual encoding at prediction time. Describe a car the way a human would, and let the pipeline do the rest:

In [53]:
new_car = pd.DataFrame([{
    'Brand': 'Maruti', 'Model': 'Maruti Swift VXI', 'Year': 2017, 'KM_Driven': 40000,
    'Fuel': 'Petrol', 'Seller_Type': 'Individual', 'Transmission': 'Manual', 'Owner': 'First Owner',
}])
log_price = pipe.predict(new_car)[0]
print(f"2017 Maruti Swift, 40,000 km: predicted price Rs {np.expm1(log_price):,.0f}")

# A brand the encoder never saw during training
unknown_car = new_car.copy()
unknown_car['Brand'] = 'Tesla'
log_price_unknown = pipe.predict(unknown_car)[0]
print(f"Unknown brand (Tesla): predicted price Rs {np.expm1(log_price_unknown):,.0f}")

2017 Maruti Swift, 40,000 km: predicted price Rs 384,506
Unknown brand (Tesla): predicted price Rs 482,988


> **Note:** Raw values in, rupees out (remember to invert the log with `expm1`). And the unknown brand did not crash anything: the one-hot branch encoded Tesla as all zeros, the neutral fallback you configured with `handle_unknown='ignore'`. The model falls back to a brand-neutral estimate, which here comes out higher than the Maruti prediction, because Maruti is a budget brand whose dummy pulls prices down. In a production system you would also log such unseen categories: they are a signal that the world has drifted from your training data.

### Swap parts freely

Because the pipeline is assembled from named parts, trying a different model is a one-line change, with zero risk of preprocessing drift between experiments:

In [54]:
for name, model in [('LinearRegression', LinearRegression()),
                    ('Ridge(alpha=1.0)', Ridge(alpha=1.0)),
                    ('Lasso(alpha=0.001)', Lasso(alpha=0.001, max_iter=50000))]:
    candidate = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    candidate.fit(X_train, y_train)
    val_rmse = root_mean_squared_error(y_val, candidate.predict(X_val))
    print(f"{name:<22} val RMSE {val_rmse:.4f}")

LinearRegression       val RMSE 0.3891
Ridge(alpha=1.0)       val RMSE 0.3896
Lasso(alpha=0.001)     val RMSE 0.4005


> **Note:** On this dataset the three land close together; with 3,472 training rows and 42 well-behaved features there is little overfitting for regularization to fix, which you know is exactly the regime where it matters least. The interesting part is the cost of finding that out: three lines, one loop, and every candidate is guaranteed to see identical preprocessing.

### Look inside

A fitted pipeline is not a black box. `named_steps` gives you every fitted part back:

In [55]:
# The scaler learned its statistics from the training rows only
fitted_scaler = pipe.named_steps['preprocess'].named_transformers_['num']
print(f"Scaler means (learned):    {fitted_scaler.mean_.round(1)}")
print(f"Training-data means:       {X_train[num_cols].mean().values.round(1)}")
print(f"Full-dataset means:        {X[num_cols].mean().values.round(1)}")

# Coefficients, with real names
feature_names = pipe.named_steps['preprocess'].get_feature_names_out()
coefs = pipe.named_steps['model'].coef_
top = sorted(zip(feature_names, coefs), key=lambda pair: -abs(pair[1]))[:6]   # largest |coefficient| first
print("\nLargest coefficients:")
for fname, coef in top:
    print(f"  {fname:<32} {coef:>6.2f}")

AttributeError: 'RobustScaler' object has no attribute 'mean_'

> **Note:** The scaler's learned means match the training-data means exactly (66,386 km), not the full-dataset means (66,216 km): direct evidence that the validation rows never entered any `fit`. On a random split the two means barely differ, but the point is structural, not numerical, and the same guarantee will hold when the split is not so friendly. The coefficients read sensibly with their names attached: premium brands (Land Rover, Mercedes-Benz, Jaguar, BMW) push the log-price up, the budget brand Tata pulls it down.

---
## YOUR TURN

<div style="background-color:#1e293b;padding:15px;border-left:6px solid #38bdf8;color:#e2e8f0">

<b>Exercise: Cheap experiments</b>

Run two experiments on the pipeline, and notice what each costs you in code:

1. Replace the <code>StandardScaler</code> in the ColumnTransformer with a <code>RobustScaler</code> and refit. What happens to validation RMSE?

2. Replace the Ridge with <code>Lasso(alpha=0.001, max_iter=50000)</code>. What happens to validation RMSE, and how many of the 42 features does Lasso zero out? (Use <code>named_steps</code> to reach the fitted coefficients.)

3. In two or three sentences: what did each experiment teach you, and what made running them so cheap?

</div>

In [ ]:
# YOUR TURN
# 1. RobustScaler variant:
#    - rebuild the ColumnTransformer with ('num', RobustScaler(), num_cols)
#    - wrap in a Pipeline with Ridge(alpha=1.0), fit, print val RMSE
# 2. Lasso variant:
#    - Pipeline(steps=[('preprocess', preprocess), ('model', Lasso(alpha=0.001, max_iter=50000))])
#    - fit, print val RMSE and how many coefficients are exactly zero
# 3. Your conclusions:

<details><summary><b>Click for Solution</b></summary>

```python
from sklearn.preprocessing import RobustScaler

robust_preprocess = ColumnTransformer(transformers=[
    ('num', RobustScaler(), num_cols),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), onehot_cols),
    ('owner', OrdinalEncoder(categories=[owner_order]), ['Owner']),
])
robust_pipe = Pipeline(steps=[('preprocess', robust_preprocess), ('model', Ridge(alpha=1.0))])
robust_pipe.fit(X_train, y_train)
print(root_mean_squared_error(y_val, robust_pipe.predict(X_val)))

lasso_pipe = Pipeline(steps=[('preprocess', preprocess), ('model', Lasso(alpha=0.001, max_iter=50000))])
lasso_pipe.fit(X_train, y_train)
print(root_mean_squared_error(y_val, lasso_pipe.predict(X_val)))
print((lasso_pipe.named_steps['model'].coef_ == 0).sum())
```

The RobustScaler swap changes validation RMSE by nothing at all (0.3896 to four decimal places). That is a real finding, not a failed experiment: with only two numeric features and no extreme outliers in them, the choice of scaler simply does not matter here. The Lasso swap scores slightly worse (about 0.4005) while zeroing 15 of the 42 features, mostly rare brand dummies: a worse-but-simpler model, and now you know the trade.

What made both experiments cheap is that the preprocessing is guaranteed identical across candidates: you changed one named part and nothing else could drift. When an experiment costs one line, trying it is faster than debating it.

</details>

---
## What You Should Be Able to Do Now

- Build a `ColumnTransformer` that routes numeric, nominal, and ordinal columns to the right transformers.
- Wrap preprocessing and a model into a `Pipeline` that fits and predicts on raw DataFrames.
- Explain why a fitted pipeline makes preprocessing leakage structurally impossible, not just unlikely.
- State what happens to columns you did not mention (`remainder='drop'`) and why you check the output width.
- Reach inside a fitted pipeline with `named_steps` to inspect transformers and name coefficients.
- Swap a scaler or a model in one line and rerun a trustworthy comparison.